In [9]:
from astroquery.gaia import Gaia
import matplotlib.pyplot as plt
from os.path import isfile
from astropy.table import Table

# Plot-Formatierung
plt.rcParams['font.size'] = 24.0
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelsize'] = 'medium'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['lines.linewidth'] = 2.0

### Skip the following if you dont want to query the database yourself

In [2]:
Gaia.login() # You might want to create your own account if you want to query the data yourself

INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


#### Printing Information about available tables and table contents

In [10]:
def print_available_gaiadr3_tables():    
    tables = Gaia.load_tables(only_names=True)
    for table in tables:
        name: str = table.get_qualified_name()
        if name.startswith("gaiadr3"):
            print(name)
print_available_gaiadr3_tables()

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
gaiadr3.gaiadr3.gaia_source
gaiadr3.gaiadr3.gaia_source_lite
gaiadr3.gaiadr3.astrophysical_parameters
gaiadr3.gaiadr3.astrophysical_parameters_supp
gaiadr3.gaiadr3.oa_neuron_information
gaiadr3.gaiadr3.oa_neuron_xp_spectra
gaiadr3.gaiadr3.total_galactic_extinction_map
gaiadr3.gaiadr3.total_galactic_extinction_map_opt
gaiadr3.gaiadr3.commanded_scan_law
gaiadr3.gaiadr3.allwise_best_neighbour
gaiadr3.gaiadr3.allwise_neighbourhood
gaiadr3.gaiadr3.apassdr9_best_neighbour
gaiadr3.gaiadr3.apassdr9_join
gaiadr3.gaiadr3.apassdr9_neighbourhood
gaiadr3.gaiadr3.dr2_neighbourhood
gaiadr3.gaiadr3.gsc23_best_neighbour
gaiadr3.gaiadr3.gsc23_join
gaiadr3.gaiadr3.gsc23_neighbourhood
gaiadr3.gaiadr3.hipparcos2_best_neighbour
gaiadr3.gaiadr3.hipparcos2_neighbourhood
gaiadr3.gaiadr3.panstarrs1_best_neighbour
gaiadr3.gaiadr3.panstarrs1_join
gaiadr3.gaiadr3.pansta

In [11]:
def print_columns(table = "gaiadr3.gaia_source"):
  gaiadr3_table = Gaia.load_table(table)
  print(f"{'NAME':<35}{'UNIT':<20}{'DESCRIPTION':<200}")
  for column in gaiadr3_table.columns:
    print(f"{str(column.name):<35}{str(column.unit):<20}{str(column.description):<200}")
print_columns()

NAME                               UNIT                DESCRIPTION                                                                                                                                                                                             
solution_id                        None                Solution Identifier                                                                                                                                                                                     
designation                        None                Unique source designation (unique across all Data Releases)                                                                                                                                             
source_id                          None                Unique source identifier (unique within a particular Data Release)                                                                                                               

### Downloading Gaia Data

In [12]:
# TAP-query for downloading data
RERUN = False
filename = "./40pc_white_dwarfs.vot"
if (not isfile(filename)) or RERUN:
    query = """
    SELECT 
        l, 
        b, 
        ra,
        ra_error,
        dec,
        dec_error,
        parallax, 
        parallax_error,
        pmra, 
        pmra_error,
        pmdec, 
        pmdec_error,
        phot_g_mean_mag, 
        teff_gspphot,
        bp_rp,
        radial_velocity,
        radial_velocity_error
    FROM 
        gaiadr3.gaia_source AS gdr3
    WHERE 
        source_id IN (4994877094997259264,431635455820288128,2312821266302222720,2414888694102153088,537563127588521600,2766234439302571904,2875903332533220992,383108338321272448,384636109728592768,395234439752169344,420531621029108608,2541549062071571968,2545505281002947200,4996458845552956928,4689789625044431616,2418116963320446720,2309122753616241280,2855386170682263424,2855790657816672000,385105360675267840,529594417061837824,4920057871348614272,2747384888699406080,2555215995900584448,4920160332088386304,4703464251158954496,2553935752048977792,2528728726428345216,2807462655009880320,2364319061964016512,2544456862306151680,4993117773313626496,4906743988126708992,2319052851148048256,2522401586766106624,418491412783587200,2349916559152267008,4926464691244602752,2377863773908424448,2377344185944929152,2523840155996531456,5006232026455470848,2552928187080872832,2356519298275043200,4925740628477887232,4987578158855872384,2803596046661176576,2582335342824976768,2472557872820632320,426122397136335872,2524879812959998592,2552121179905893888,2531326283993100416,2790494815860044544,2790494850219788160,307323228064848512,4687445500635789184,2533575369771883648,2591754107321120896,2456160271800004096,320029150076023808,4983839647522981504,372111985092019840,4934162784467241472,5037084872486444928,2587268134239449728,406775841506041344,2585189473147457408,5016473702391268096,396963005168870528,398672715686799488,4912620293662031104,2457759374023232768,2484544095751034496,2480523216087975040,5139880551029408768,4617960488907213184,2589304876450784128,5012346101380568832,572487740053132288,2568588664341691520,4698424845771339520,303637562009656704,98092934167683072,5025127443016406144,291057843317534464,318528007466920192,4686800224727589248,518201792978858880,356922880493142016,297470774951165568,105240786245136256,78629306318257536,2574620550768898432,78649033103283584,5149836834977282048,2490975272405858048,92597914738232448,5020119579868434944,330661599315957504,104648944047154432,333010327952701696,522115156720215040,2486388560866377856,4970215770740383616,561231932144411008,332820971434386432,351429930856782464,458558784733311232,4963617807621683712,355669578975070976,455517329408362752,2516606022320239104,457474219590579328,87648226538760064,5068532996689788544,515289392829738624,459237630076876672,85787470611883008,2516322146457318144,19693180966870656,131188715200383872,127366331745660288,5146358426863612160,88996326578456192,2488960249844340352,5125186299678747008,457817164137702784,5064259336725948672,2502097283492466560,452244667407804800,81606375784491520,5070533180139377792,126342377182577408,133116227803380224,25405350031335296,4725208704209795712,4725152392894633984,5128633195616949120,2495751967528809216,453562088496088320,139623068897753856,20484382662003968,5077717389114829056,4640029027306103296,143076256963396480,4696032750850438272,439494077735062144,4645960583300470656,5065010887285008128,8578256576520320,4618550411255512448,6963383233077632,4626680917489564160,466384799259137920,5179920984941752448,439905192004402304,4646535078125821568,492442090962517376,5167424240722533760,5181816233750415488,4672306015773211008,4733604373137759616,4850844228558819584,5167510797199003904,235842052999634944,4620280217923548800,5058635403471767680,4613612951211823104,4613612951211823616,239721228805415296,234469931207274112,62884540326096896,3261591090771914240,54253618862057728,4723118914857785472,5060587895604134400,5053839127592396032,3264551560189562112,5088251195844051328,4860061541910268160,63126196662620416,4829340465475546880,44901791432527232,3249479592234269056,3249479592235301376,63054590968017408,66837563803594880,3270079526697712768,243645699341060608,3251244858154433536,4667446586695129472,3302846072717868416,4683172420470864256,244214799689691904,53278867446391040,3301319572621418368,250862824946594816,39387328302768640,66144699680456320,46896958359881728,48829075167625472,3189621320226676736,551153263105246208,5090109228757394048,3195919254111315712,4678664766393827328,229143725086190336,4782553840532147840,3191738120628213760,4885878590326674560,232990572675079296,470639806179201792,4789516154317811456,3307009304776119936,256832412872262656,3283853143218651264,4839901583898106496,3198881613315343872,3202808828330265088,470826482637310848,2978374522004181888,4864752883148064512,4867574023826934272,3282093301842469376,4891567154251463680,159277625222514048,4677461862018523776,151650935831913216,3186021141200137472,3292685210887016192,3173236054352419072,4815192671404177280,4893995356962398208,199060743352070528,201881364339401728,3228023859770327424,4764692171059541120,2976789094955826560,2982808337003815040,283928743068277376,3392181976588782336,2986577153625081344,2986473765172532864,3242153305741855744,261664427174056320,3422405214775411840,4652123895783242112,3209308900555979776,208021900557436800,4805691447831511680,195470288131984128,263082591016645504,190802650815160960,4769136809374612992,194394347281717504,481698110012697728,197048460976343168,190998058945380608,2967199704297025664,3455921181049073280,3429296884940000000,3337021260634868224,3346603611845335424,2914000658819492864,3218697767783768320,3011223668834627328,3022956969731332096,198027266851070848,3346787883122375680,3320184202856435840,4650712810002383104,4767805506952137600,3024247796382652800,3348678699526809600,3329569015639064192,1114054838014073984,5482551252566796928,5210778263380292096,1007682723024253184,960039814744267520,3324181683539044224,5573532025833121408,2940238304793401472,3117320802840630400,3382277296672027136,995112350178946048,2895487456076198656,957295438016687616,3383000470383852672,3103811515783541504,3326650224581677312,3386162214851684864,2947050466531873024,2925551818747071488,3126453655659835136,943770757800160384,2926944659464115328,1115546944012493696,3358418684623972864,1100655330324351616,5564029702750970112,5564028981196462336,1100267237077449728,5564171814627287296,3129655296079487872,3112162508466471808,3160815489969826944,3052418589963163264,890661253803216896,3052844272764398208,5480556532316697216,1140292483988569856,883006556929519872,980918204822332416,3051506991743533184,5504790696302830208,5280944182024023552,946030529073021440,3059515898856688000,5589354543620145024,3156974449176326528,974375354722176768,885100916128136064,5605383430285597696,3169486960220617088,3032020450247993600,3166841329084630784,983979721933760768,5510964144860346496,1089400763661597440,5536077746353130240,976040702520790400,5717278911884258176,874900643675606912,5588614164276695424,926002203218885504,3151407827963355136,3144837318276010624,3144837112115126400,3080914632816248448,1084764020047940096,5273943488410008832,925532265076649472,984190381489031424,5513896164414899456,5602379190880667904,5698587862747531008,5725503293214289280,1081813343155426560,922434906461782272,923023248261420672,933287876501371520,5597759970724418688,680099824985004288,3145802036649641472,908962109449446656,5274517467840296832,5320436784269889792,905923368548465152,876571935709366272,5290126272348542080,5544743925212648320,657056745624156416,675615677264385536,931573222477949696,5516345223493485952,676031052142751488,5271072526109138176,5548080118369905408,5723025990435533056,5272690766709543680,5753337533146097280,5322552760038496512,710766750855439360,710947968027922176,5709868512742196608,5748802661860766208,5322090003089341440,5639391810273308416,5734737438536674432,690287663506579328,1042071701528617856,1042071701528617728,5625513014289760128,717393648787722624,716504796716020352,3073747187092773632,5302618648583292800,611645983387812992,5652718097353105664,5652689406971618048,5755957119598921728,1117334024067935872,912718071240545152,5764485618978306176,712888090655562624,715357559411376128,684598618544000384,5620763437599189376,5762406957886626816,711744456827031680,1123700235048742016,5648566371510973824,5624029566946316928,611074413433751680,5651964996310470144,688606063549945600,5649808720867457664,636732410620758016,1040160612880069888,1022780838739029120,5427528254746168192,591040864898749312,5297051821220130560,818602457173176192,5423545067716723712,5423896396040719744,3843957354387777152,700531568527365376,5314177402013456256,1039163458912998272,633911196927706880,5219215228420645248,5681903877597244032,5633102260158519936,696261653777188864,5436014972680358784,5436014972680358272,5432789383518999168,826275295988399232,5688717241916173568,5425208663166556288,5410698683096655360,3820383996887145088,820105214691139584,639665392247496576,643183348419998336,1020653077580086784,616396182856185728,1066726497434084864,5242316444456199808,642685200933153408,615733593956317568,808030648576069760,3828424828500179584,3822028007288795264,1053211437944753024,5407613384750351232,5473389537569200896,740483560857296768,5669427512997660800,5356710428806242816,805470233890349440,3875651975353757440,3875652014008894720,5446665014103742336,5446784345474819968,1146403741412820864,833470465721350784,755877620910173696,1052520154368111872,5441714531716273920,3882611201058534400,3855631797052747136,3883918657822146944,5469171261208778240,3750749378584132992,3754712881779364992,5233071514480420864,1076941716370493696,5367774809996936960,3553682127126319360,3870354528331257984,3884899559633556224,5231870985222281088,3554395813252626048,5391794195555570048,1055313016981775488,776762672481353984,3557394009663078912,730459763934332416,5353122722363361920,3549471753507182592,3763445409285757824,3789156870225942656,5456685104084844032,3982007636324256000,777395029106166272,776981269136616960,3552845261339955200,3968635204109066880,861050512312844672,5401688425816913920,3788194488314248832,3818473629793533312,843807902246527232,3990494251184010496,3810099989754827136,3817262208497857024,5376644127919488896,3790040465258127616,5374565879145559424,3810933247769901696,3783206210217512320,3978879594463300992,3978862277154958592,859082970614616448,3482983495102507904,5398247534240054528,3478127467639543296,859567752163281792,4019458647338779648,3585097235918075776,3464893058489831552,3793132257595347200,5332606522595645952,863131372427958912,4021565827014024576,3794567429507510528,5224999346778496128,5377849123945711872,5381346739148118016,3480776843983381632,4004185576130620288,3490527755479959936,1058284412796260480,3487220772397809536,3479615106870788864,4004395720290994048,5334619419176460928,3920187251456610816,3926968661219149184,3795052348495488896,3891115064506627840,5341271911184522880,1537794524729363712,3575728709655386752,3575770010060921728,3489719481290397696,3904628406009296896,1575357587146077056,1717341818608187648,3694399755554510720,6151294355090597504,1683330453627242752,6054148143441683072,3905335598144227200,3947104533054775168,1518943638389520384,5860131207828395648,4015547856277853824,3583181371265430656,3696778892558087552,3583402849843287680,6127190796769848960,3708578473389974528,6133033635916500608,1534384148897669248,6127333286605955072,1541286711100812160,3935942939548822272,3500086050578451712,3528871819044810368,6158704208764041472,5784295623056963200,6079710620510474240,3530520910392199680,1570514066627694336,1517239773324267904,1531097433767946240,1460689760003983232,3704392873141718912,3690395231125833600,1679365202380970880,1459546263999675264,6154946657841163392,3629758603668233728,1686322048672412288,5784706428090844160,1726678630833373824,6085402414245451520,1566603962760532736,3691685882071367936,3506567328028533120,6067215083178616704,1556005701461744640,3623233040812235904,3943650619138622848,1688618481786030336,3506061587037686144,5787859896160384384,6063252374582712320,1551062778220116992,5869567658943170048,6188345358621778816,3630035787972473600,1468213546275054208,6087659745978472064,5845312191917620224,1552488776081383040,1472029470098019072,3713594960831605760,5850533227210203520,1686708050268594944,3714914271705535360,6165095738576250624,6165010320266420096,5851860818806598528,1500607765872799616,1658578797618935680,3725570772761744384,1251824057289839744,3728074738695246336,1451566149955227520,1502063317405058304,6177238676273826304,3727155340815968256,3618657732410663808,6178524211524592640,6275184065428686464,6114344102909485824,1505825635741455872,1671668067321674368,5846206030463663232,1454347089739329152,3643555726544985088,1498447607777405312,3667514634669861120,5790182751914903424,5867776696271127424,1232045934759720192,6296317052576778240,6118079453145005312,5852538324131652736,6329136310728635776,6093257119157372160,5898935893701856128,1176717792385803136,1507571286545131648,6272326022391660928,6271903947364173056,5772718006135360128,1493367245581725184,5893318763718868736,5785654859950570496,6310804634396281984,1282448170543051520,1282448170543051648,5799644049485006848,6282457918962299776,1281989124439286912,6227687980608264064,6332763530870415488,1620030637207462912,1276688069644366592,6255777749623059584,5903884280152869632,1722236328978172928,1645204475617697536,5902612969841664768,6206195620664214400,1602197696772907648,1273685372108354176,5826604589994061312,6265877455415860224,5827454649955072256,4424031479858305408,1401010605309570816,1193520666521113344,1641386833807056640,1370654257498865536,5985749857190372480,1218051664291152000,1376326912864142336,6009537829925128064,1598771000065137536,6008386881767536128,6008581907636919936,1202826348825240832,1403348682426861568,4348098485293072128,4425632987265111680,6264127170346899712,6250213984568447872,1404831472640252928,1372458109403442432,5998095590373665792,1622697949337226112,4349513797276615680,4341773063622911872,4411572123331402880,5806672265237931776,1647162396588999552,4341495230772911616,1323922779935068160,4458207634145130368,1385719147346936064,4452997701376633728,6246049446837287680,4355229123137665792,5931881969426463616,6018034958869558912,6022366686772364288,1201036206454896000,4452521234885949184,4323956302321933952,1331106782752978688,1653044367185115264,6018613473779340928,6044265144466741888,4440264291578812928,6018693257096471424,1703379704562897280,1431176943768691328,4466388790929771904,1405343643196929536,5765270154886903168,4435778215414219520,1326398777041821568,1425909733315616000,4462612140287443840,5929529014509678720,1431783457574556672,4126670518631322880,6027138331058857088,4379328051494006784,4334641562477923712,1631578537252535040,1351956512512484480,4125468645047515904,1306197930941817984,4565048312887877888,5775307733975564160,4340322876499078400,1358325021299899136,1313405848136604672,1358301480583401728,5765288880944438144,5808208420421074560,4444590625015876864,5923906524348891904,4380188694219651456,5938773880045035776,4139348334376604928,4379812558164849408,1408135749896104192,1408135749896103936,4573071139998034048,4108828945319007744,4388138816124225792,4139531467491239680,1638563322306634368,4361621688038664064,4136103572502555264,1341543072245722752,4359722208685335552,5915797694789556096,4387171623850187648,4367353266060378624,1336988963803208192,4491748511228743808,5975317695158795776,4598266758185956864,4542785981266940416,1706631093589103872,4598775557191664384,4110161965722210432,1348795004265872000,5921433963191695872,4055353888055627008,4168312459956062208,4053455379420643584,4581383928942599296,4118171568655203328,5909078751020702080,1349256249394224128,4549622027311531904,4500646618315862144,5961193055261256320,4117081643422165120,4149820323678136192,5909739660590724224,4150020774057837440,4596322473734130304,5920900901901635968,5945252202546434432,1638979384378696704,1350517492310907392,4068499305485306240,1711005951573009792,5767614206300408576,4604247070649603584,4067477554248982528,4611543459874840832,4499839473701254400,4611204329256349312,5911160263973498496,2123432247756223488,6363592840482769792,4037085334973293824,6414612172876713472,4470233817461336704,6431530976766770560,6725656144031366144,4590489981163833984,6345857271249415424,4153937891610652928,2149331587745863680,4578913738632417920,4497414466452138496,4146666271458052352,4094555467661923328,4523585076572785408,4585067258532443776,2158285185808357504,2149253075743572224,6635520414133666944,6437614815119680768,4154063678315488640,4528439381757452928,4152557420406043264,4484277256704949376,4484289866726156160,4484184592790777728,4483974792231866624,4257461275049675008,4257063458004688896,4257063453704172416,2150504594853811456,2118649750133781376,6432020637402543616,4589139574728058880,4525569007873380736,2256410856215182464,6709854989379725056,6651133479247436032,6631915390382416256,4107012041007171456,4280632829779587072,6730516225919672704,6705845452725619072,4512265810525783680,4539227892919675648,4504577986571873280,4252064631569619712,4203108841963846016,6761581964903088256,2292861388958880640,2146645790077864704,4073522222505044224,4518917168694695168,2092134443120038528,2146576589564898688,4088653838978654336,2262849634963004416,4254454797960984192,6658005048964693760,6718079679950008704,6763036202147206912,4268167357267580160,2140481412496465152,2039140284770609152,4320094439580536320,4320303621677848832,2295446546953958272,6664729284121316864,2052891361294411520,2127093140445053696,6742379406616165376,4201781696994073472,4293873732939569920,4319908862597055232,4288942973032203904,2018864362679341824,2127566548919332608,2142307563871222912,4213409341688406912,4314903198513595776,4215241712185612544,6744843579679059968,2024985481361040384,2025389380082340992,2248748668919802496,2301882675705225472,6740493675455190784,6671045050707117568,2080526555267049984,2080526555267050496,4181798519823760256,4298029268399256704,4240231824768647040,2078430778727685632,2073772770741915264,6368021054841781376,4235280071072332672,6666783962114636288,6670827416123874688,4237555506083389568,4190813690536580608,6673731810450381056,6865904860773722496,2237893023118101504,6424223313252632576,6874124023727679104,6853784501721502720,6749419923164242816,6853523539508720000,2053953008490747392,6443365570172526848,6672244029484054272,4249667902270614272,4230380819051252736,6361559602963567744,6692573110423893376,6879524790480960896,6693352488073255936,6429048245152936320,6797171060323993728,6680097978480311040,2185261016407220224,2302010356492847744,6468965052723781888,6679105566157898880,4217793816094052480,6429465338016396928,6875432476922523520,2064284054100290048,6862687522250677376,2064689567732385792,1831553382794173824,6683212727417918848,6857939315643803776,6681773947733560192,6424566979354709248,2169971345155578752,1871118140493076224,6470278694244646912,1844125748497557632,6454251250683840896,6805808514433280000,6805792571514600960,6808651507904773888,6857295585945072128,6913810483611035776,2188860027203347968,2169025009235266816,6478328218869704192,1864760695541016832,1789361097242243584,1841254644460354688,6479860113444922368,6774018369099513216,2690697646876721152,1737167215848315264,6912866381081015552,6831993452567326592,6788656957673130112,6348672845649310464,1739921801713625600,1841683423932168832,2176116580055936512,6462911897617050240,2191146977029443584,6791196382856581376,2685959542034846464,6583325635088476544,2693095097621419648,6578917727331681536,2177776331525931136,2274076297221555968,2300234782654298624,1797472370615364992,1951870157081161216,1977417206686680704,6842831888437047680,6585792870460638336,6578616598584224640,6844375121726139520,6589369272547881856,6592315723192176896,1792830060723673472,2701893698904233216,2667464656943675392,6584418167391671808,2696628687474414208,1800298527816525824,2669936427801840256,2202703050401536000,2693940725141960192,2679976510857026048,6617996741403360128,6558209044297239424,1892992267183979776,2676567307551465088,2676566272464334720,6412251624488840448,2199371701965748992,2683452758602312192,6358158435541361792,6613289285448236288,6397887600292497408,6409446323650635264,1900382604528495744,2735175263041913088,2198431172852758656,2600033326799287296,1955134710179436672,6573778541262656256,1907041590544054656,2006217676803960960,1999615350008375552,6404417771644764160,2730707260103011712,2205493129867600256,2730508416002618752,1875613386395668864,6357629089412187648,6357630601240673792,2736054627915080448,1875301369907249024,6505318682415051520,2733055335904034432,6505113009316709760,2628943473222829440,6520844168856463104,6506903598362207872,2731866347221858432,2836609093355562496,1884744525522874880,1929287700069881856,2410908771246898176,2286958798223194624,2611561706216413696,2610488514148351360,2711324446359728384,1989342372349280384,2208530698238308736,6554977369168846720,1936315366080098432,6552878165248320896,1929838143078434432,2842462137347560320,2386223463892847872,1996725077535283200,6387649708219253248,2812250821990695936,2631876970245863552,2638553754605793408,2407167579853954688,2818957013992481280,6527675297155337472,2631967439437024384,2405805697263561600,2869130517001766400,2813020961166816512,2814629409239942272,2660358032257156736,1923682286712356992,6389551313580046464,2865535629374939520,2395444208921491456,2439184705619919488,2393875961742886656,2826254713186397440,6485572518732377856,2394366515727615104,2871730307948650368,2763719512612656000,2742789930821144320,6531195177474061824,6521660556236440960,2867032958053059200,2448933731627261824,2313836325604479616,6350786278796634112,6521875098442800512,1921351390779081600)
    """

    # Perform TAP query
    job = Gaia.launch_job_async(query)  # This runs for <30min
    results = job.get_results()

    # Save to file
    results.write(filename, format="votable", overwrite=True)
    print("Data download complete.")
else:
    print("Data file found. Skipping Gaia query.")

INFO: Query finished. [astroquery.utils.tap.core]
Data download complete.


In [13]:
# TAP-query for downloading data
RERUN = False
filename = "./100pc_white_dwarfs.vot"
if (not isfile(filename)) or RERUN:
    query = """
    SELECT 
        l, 
        b, 
        ra,
        ra_error,
        dec,
        dec_error,
        parallax, 
        parallax_error,
        pmra, 
        pmra_error,
        pmdec, 
        pmdec_error,
        phot_g_mean_mag, 
        teff_gspphot,
        bp_rp,
        radial_velocity,
        radial_velocity_error
    FROM 
        gaiadr3.gaia_source AS gdr3
    WHERE 
        source_id IN (1268321022907264,1534510211124352,1559111783825792,1608138835023488,1843503042736512,2021898804878976,4719859820691584,5526420319283584,6407197852445440,7027562929324416,7158988928491776,12254267546754176,14862584004642688,14895255820376064,16166875377576064,16166875378033664,17033153101264640,17709047809907584,19388307008211584,19399306419780224,27643135366679936,36204856318593280,36207059639735296,36918203143996544,39124751182907520,39387328302768640,41026764562266368,41518658576469760,42871199614383616,42871268333859584,43129206888307584,43755133947362816,44134980855106944,44478578238260864,48522895539002624,49013483883923968,50456318016101120,54713901916661376,54993654611672704,55103056018558592,55783726731081984,56010088686601856,56048945255703680,56621554593127808,59077760488621184,59521580936483328,59726644147616384,60007152756147584,77708843286683904,77986298173874176,78629306318257536,79950305114153472,80107603996819584,85008707141970304,85469849190835968,86082208448194944,87377467505518592,87648226538760064,88996326578456192,89061369563264256,89512001827263616,90076974709491968,90367383218249472,90565849362482432,91460813172927488,91690164426711040,91980130553425536,92009503837556096,92597914738232448,93002813806259584,94616892580412032,96095735719745280,96217820165602048,98035961426035712,98075028448822656,98092934167683072,98148287706192256,98381792193298176,99064833727036160,99498964725981440,99915890086770176,99989797884119040,101673523847748736,101854122928108288,102038286830643840,103440095437050752,104118528470334720,104648944047154432,105240786245136256,105430868613160832,105716359383595136,106241174322991360,107794440655024896,109247788869176448,126342377182577408,126972461769858816,127366331745660288,127429927326231936,127700269747929344,127856640916806528,128707181880407680,128826375813389568,131978710009614336,133116227803380224,136540274515184768,136610338318092544,136692075844561280,137141810455464192,139623068897753856,139937559287208192,140253260859027200,144039905890556160,145429108768875008,147095826660107520,147661731550958976,149884909703520000,150260530363309056,150371374878785792,150577636388844160,151015826131149440,152371871863155840,152931141027253888,165066141626360320,165401149074789504,165677710612522752,166091096919353088,166587938734739328,172616247455056384,175817750438186240,175817887877152896,177368203568437120,196617731593326976,210000437370798976,220421260684140288,221424221446885248,224218080493203328,224331059611503232,236921601622832384,237144321446511232,238653229356266752,238701332988102912,239721228805415296,241335036997623296,241745640169579776,242802266545591936,247739176832560384,248346828806376704,248937133409693440,250278709394343936,250422191362602112,250862824946594816,251657389598509184,275550743638838144,276192136878054656,276438530560644864,276576484910598912,277090540953615616,277138090534130560,279056600887092736,283096760659311744,283881120469767936,285659030771435392,286251461381258624,286315576653291904,286925049691578496,286977585732431488,288744844514936576,289974716990002304,290222381984000128,290531967521897856,290602027028411136,290678477446232960,292454841560140032,292790566268764288,293138458619500800,293467178236290432,293639457964733696,294023805998116864,294798377579784832,295779073232577280,295831781071536896,296317765211001472,297399444134124032,297470774951165568,297812418125109632,299265624604662656,299563420457225600,300313218667865344,301521311363899904,302303987548932736,302388929117252736,302740016924068608,303508438112954496,303637562009656704,303859968300444416,303958443311053824,305120700116366080,305232742928041216,305977799494481792,306350606950880128,306779618349361920,306805388153226368,307116824821161088,307246738992334464,308012445761585024,308383019835259008,308651262017484544,309659892137143680,310420410586857856,310420410587121920,311899773416896000,312589747027492736,313521312549882496,313967881773971840,313996228559363840,314637484355364224,314956480167069696,315687277442204672,315800325276303872,316274416650957952,316773766729205632,317317170285551104,317446015009891328,320029150076023808,320903154445635456,323578506753737856,323578506753738240,325134624944612992,336619023898043776,337298663818927616,337449812306056832,339985316184848384,341432690100603904,349026806460052352,352179415533582080,353783465558846976,353917571618022784,354102083412499712,360858960322547968,361418474306342656,362287638243286400,362877874124807936,363066814030170624,363710101347919104,363756658793281024,364978005758397952,365204711311382400,365553737534259968,366429258027943040,366784816895496064,367799116372410752,367873844509139584,368075574827351168,372111985092019840,372147336967326080,372516807234476928,374811767174987776,375131415820709632,377196947196848128,377231345590861824,377431181828043136,377520826387065856,377790511677100288,378856389416637056,381396329995329408,396370097820352256,396963005168870528,397957277212371072,398672715686799488,399602662005852544,400329056939182976,401215160231429120,401316766273630208,401529105161219968,401734232798946688,406267557895785728,406775841506041344,408556255774161536,412839403319209600,433533998859654016,505759238365541888,509163429445659008,515466135029128576,518201792978858880,518327583983793280,520790043346964608,521406968152848640,521927243309686656,522115156720215040,545879107688878592,545997618721866624,547179765520150912,547501815051141248,548423278811480960,555223753938122240,555258461563927552,558036098518042752,558080903616798848,558153333945938560,558285928175703424,568089105130012800,568149720002755584,568168544844912128,577026966432521088,577035178410047616,577342113952420224,578452448897807616,578452453193409920,579835153490062976,582729312907277568,583533124626862464,583995091309178624,584402318633098624,585164761227127424,585488567401589248,585513959248023936,585700189030449920,586075122495366400,587767202170593536,588191824112710272,588349771535238912,589285627728700160,590183142750379136,590316282441032704,591040864898749312,591207406550605312,592019425952313600,592605431290049152,595617852632038144,595867686585664256,597646245427109376,597819555947490432,598240497102033024,598769843231677056,599820460951495680,599922307511771776,600139357980010752,600297481495526400,601566038739612160,602669506032575488,603025777864658176,603155765049434752,604127802048700032,604972428842238080,606216148291719680,606938634805314816,606938634805314944,606956776747174016,607535218647587328,607977291041540608,608311336417663616,609236712891803264,609898313949732608,610478649930041856,610896872370416128,610906974133485056,611074413433751680,611401999180118528,611645983387812992,612262122214240512,612781160422662016,613553056239392768,614379236149119360,614643325098055680,615705144092773376,615733593956317568,616394263005643776,617094132222221824,617329255911052416,618571635330439040,619658708733339008,621824987157964544,621979502901486720,622536749138267904,623384098941292672,624090737025312000,624510170646474112,624600566824086016,625054493327257856,625068134142899968,625075796364911360,625415030061614336,626112262167369472,626319721972908288,626329892455217152,626844326458669568,627240597321087488,628141337862068864,631422383638520832,633295848373672320,633537122455925632,633911196927706880,634225902066816640,634307433430523648,634839219101579136,635472537799028096,635777617916154368,636189900416251136,636607989713082880,636651837034002944,636732410620758016,637527525030988160,637716641031134848,638935758908460288,639439884989423616,639665392247496576,641362724667841536,641502740602215040,641625576666483584,641625576666484480,642463713764824192,642837135401004672,643183348419998336,643650361688870656,643989938983182080,644705171296887552,646514250177090304,647899806626643200,648307961663426048,648795766869070720,649041786893358336,649129777885760640,649672906567630976,650230771279461760,650340408909616896,650400263572196480,650432664805355392,650631470251495808,651337945128014720,652195319383139072,652438247031965056,653051877600013312,655245124121246720,655355968638552576,655808799925588992,657056745624156416,657598147723622912,657673498630210176,658781771991055104,659660006903352192,660212752014880256,660222170879118208,660431181166277120,661504265860547200,662043992928302080,662716271271960320,663601481211006592,664021842547278592,665392383790872192,665991789427730432,666111739274159360,666207740383536640,666999556257654784,667811442514553856,668127242870832768,668317840636534912,668796162553518720,669024203839693056,669444943132791424,670842761712806912,672816969200760064,673167507250782976,674429853974804352,675615677264385536,676031052142751488,676773359928988672,677060225090183552,677196843705878272,677560232299128192,677593660028220544,677818750673797376,677874722688200832,679311750025779584,679723276611814400,680099824985004288,680152090443478656,681595744913809280,681934703732973824,682155327614998272,682315757527779584,682597056409463680,683959213878018432,684152216824047104,684258010458273152,684598618544000384,685295640197734528,685402357248747648,685671909396428544,686288018160731520,686867284694206336,687576710212105856,687914432080614528,688157703323842944,688314207636275328,688606063549945600,688672240406772736,688818750329816192,690287663506579328,691116660913783936,691600338653939328,691604221304319744,691608584991130752,692130376274309376,692134636881870208,692134843040270080,692904776058982144,694006757290271360,695066274182696960,695630598526264960,696016634481430912,699487479157292288,699628147926656768,699920381797443712,701307621874580864,704316671665447296,705484593532621952,706092898340402432,707022569781407104,707408841962407936,707673416242396160,708292166410436608,709004920527186816,709051168734737024,709769115467848448,709798939720824192,710026160671981440,710267950150378240,710706346051677184,710766750855439360,710947968027922176,711234420874380800,711260465552649216,711527372000026624,711612103114782592,712789886226437632,712888090655562624,713438705462521728,713654106662213504,714097617868691328,715357559411376128,715437789400182016,715502037814544384,716010802461047552,716112442863454464,716504796716020352,716743042845256576,717044652628561408,717203467636317056,717762977319125632,718682620012682112,719707193051500928,719903902553512704,720353602808918016,721244821406496640,722391440239688064,723041560844852096,723319324969842304,723524383888419840,724966844360175232,725741342927857664,726057456815979904,726160806613850624,729219716681441280,729726763340729600,730264772419060608,731177607588663552,732841233105592320,733294781652279680,733902090027573376,736131388507504640,736984742674902912,737619062099922688,737721346745881728,738060065046666240,738102190085868672,738375899762150912,739253825436690432,739337319605413888,739577567186823552,739952633796191232,741436871415359488,742698702740757632,743174168505385216,744878171010647552,745127553991482368,745137591330297728,745895322345421568,747348704918599552,748517142181920768,748521544523051136,749766221750309632,750160156150755328,750469737393655936,750713313579071232,752524965144307584,752673674091348608,752814274140797568,753840603821390848,754853769426521856,755466845943050368,756515161560637952,756823437133003136,756921259308136064,757039559887727104,757272896870658176,758449752269796864,759949799663166464,759980585988726912,761213821423029632,761336623131952128,761480972683472384,762229770166576128,762295264123316864,762520556632292480,762670262012752256,763197172895543168,763471702912929024,763893885310438784,763955591105229056,763981296484951936,764086604787758208,764725833360592640,765404128955927424,765628876004078464,765942683495024000,767072088095193600,767724270288678784,768139786900195456,769375702394078208,769543893313272576,770539191854568448,771045212017101952,771517005584473600,771585720766099328,772875173027382016,775531009004593664,775677140971643776,776981269136616960,777221198894621440,777376848509180672,778326792196212352,778707154499553792,778845108849112704,779223787525884928,779588657882981120,780712285751716224,782378320745502336,782910557397183872,783880704600816000,788171510073699328,788619668448368512,788768476168350976,789242022082351616,789339608048607232,789712823515276416,789765733214502144,790014605097236224,790770583763244160,791138993175412480,791223891790995072,792275437224230016,793309222967482368,793575201703625984,795686191013196544,796546150842518400,798566915774818176,798602267650286464,798602271945568256,798742730259141376,799278226781362048,801407190530270464,801512095107318144,801684301820413952,801851053926004608,803211596486728064,803280861423538432,803585018126670208,803647514191465600,804230637613089024,805470233890349440,805697489201014272,807222713688595200,807280785942808320,808030648576069760,808268967721645696,810478814294621568,810719091945103616,810799665531428608,811059120212556416,811237923994639744,811435114533149056,811655223016756608,812202337427426304,812367843990691840,812885610888282880,814576110016081920,814676062494786816,815084565424733824,815163352304154496,815429197896495104,817371412171264512,818602457173176192,818833831355548928,819574013135499776,820105214691139584,820169737983275904,820542992117501952,820663521783083776,820899298308206208,820969357814798080,820969357814799872,822352371643412352,823153056627033472,823393402996648960,823631692078674816,823804585985883776,824451339341466496,824743882449610752,825049851623687296,825112523786586240,826275295988399232,826954824238274048,829441717689202816,830202545366176640,830333490329229696,830472368094495232,830949079408842752,831349748309446016,831390705119554560,831946229073235200,832162626703637888,833452873535333760,834234385783616640,834901991204579712,835731057331866880,836823151552466304,837242890116333312,837941213142775680,839750257663742464,840539672652621568,840622479622035712,842231163916746240,842469143761181824,842498482680906496,842504358196167680,842958082837662080,843807902246527232,844579995631363712,845666931600086144,845973046799769984,845982667526493440,847003323555811328,847272257228823552,847884509109410816,848290091460808448,848427427335438464,850296219145725184,850558899345419520,850605735964206720,850713449448779008,851386110046262528,851682875106449664,852225282231361280,852433880203843712,852604201427010816,854072904147710464,854220964556656768,854393484803051776,855361055035055104,856770590286297472,857165662854257792,857552239974460160,858651549739624320,859082970614616448,859567752163281792,860312117239402752,861050512312844672,861438223304637696,862185002154275968,862260868456553856,862946654474456832,863131372427958912,863215004030398080,863418722918878848,864108971408318848,864111582748403968,864943569460727040,866081701427025536,866574493091351936,866850569292962688,866962856917166208,867418432688616448,867440427218037120,867739936760402560,867848380391409408,868706888517664000,872009447786700672,873303774834413056,873994719110827264,874900643675606912,877134026669419520,877161789337691520,877486458802481664,878635723331279360,878933862779706112,879383936697674496,880354496226790400,880840789603989504,880938921016895360,881146316398060672,886366114708208640,887304268709558784,887534135357644416,887758130788405504,888152408786189952,892944179243357056,893633710472340992,893881581625207680,894591075862208128,894957694270675584,895351903549937408,896326002132552320,896951830406797568,897217706062573568,898002379406954752,898074225620368768,898348313253395968,898473932456801024,900182131145007104,900399039877540096,901360494077262208,902414964384353408,902857685316692352,902972034525449088,904376149529962880,904604259537257856,905356016253570432,906160863060179072,906775593140073344,907000821224488832,908048587085834752,908364559240386944,908455372028740608,908716364305366528,908957092928687104,908962109449446656,909928404076399744,910087768838947456,912559466682042240,912665947516962560,912718071240545152,912788233820354560,914211487199883008,915672081021495552,916276228300814976,917321936874346880,917707418778058368,918237791404732416,918336060256260480,918554863070270208,918648974394418816,918886709424112896,919692273490212992,920086173530528256,920486606217458944,920673244020385792,921281480108851200,921642532239821440,921804126089222784,922164422305158912,922434906461782272,922686832063591168,924219890574729856,924428973878531968,924478859922238720,924755185235078144,925003537421986304,925258860342797568,925532265076649472,926002203218885504,926374594063195904,927205339521671552,927618687178189056,928079146323608320,928440365957680256,928937074630507392,929196314561278080,930242465515189376,931433752004161664,931500512976150400,931573222477949696,932855386179190272,933057593239584000,933095015289999872,933287876501371520,933421359789700736,933544818624946048,934555334235313664,934891926527276288,934937139647967232,935584202240404224,935585340408437888,936641833642799744,941091346746943616,942461445610425856,943770757800160384,948132103814834560,948173369860748160,948246835277407744,950361883331847424,968662184929578496,973275770078523520,976027607164699392,984190381489031424,1010300965150157824,1011466005095102464,1011870071322756096,1012210992942502144,1012271221268203136,1014142383899931392,1014793672740325248,1015768939554753920,1015799828959492352,1016560970179771264,1016617865610081152,1017803276584446336,1017868869324495488,1018214665731836032,1018659143306715392,1019874314110122368,1020076452450631296,1020158400426591872,1020653077580086784,1020868302685710848,1021408850089922560,1022470261063639808,1022780838739029120,1025040880593486720,1025090152458148096,1026185403478262912,1026249243872649856,1026514226174393344,1027695136022450432,1027704894188127232,1028072371590379136,1028955489881696768,1029968414968983424,1030289712881895296,1030381796981255936,1030621658018969856,1031832946170531456,1032545631568349056,1032793743239555712,1032933449935852416,1033040583599433728,1034060277555338368,1034473453408991360,1034734243823264000,1035126803834023296,1035222289547372800,1035772904354245120,1035806851776104448,1036295206737529856,1038018828652951296,1038076351151065088,1038360708050042496,1038500174228188416,1038689805623461376,1038928502726153984,1039078998380506880,1039163458912998272,1039435691120199936,1040160612880069888,1040866941726072064,1040892574090944768,1040991259555635328,1041763971415231872,1042192506072573312,1042926292644833024,1043885758275341184,1044109229717304064,1044511002434919936,1045463042064412160,1045634389784799104,1047132925349510784,1047228922162323968,1047653608528684416,1047912479093335040,1048769578471443072,1049324629980448256,1050272718241775488,1050402662475992320,1050570750316642304,1050825733935333376,1051676171819127040,1051954485699665280,1052563683861162880,1053106739526215424,1053211437944753024,1053321934567306752,1054193507986658432,1054610222893247104,1055273537642488576,1055313016981775488,1055533400343235456,1056003918305358592,1056520418188043264,1056993173828693888,1056998259069523584,1057020382445488512,1057653804222121600,1058284412796260480,1059423708705441152,1062419637373273728,1064966106303092864,1064978578888570496,1065088220813167744,1066726497434084864,1067421801098857728,1067716813814487296,1067799822645432192,1068317142866278016,1068706339918614016,1069735341067492224,1070129451562454400,1070600145618332160,1075122127704964864,1079502577736051456,1080170840286325632,1080303919849120128,1080678230543387648,1092515229130044544,1094608119452121856,1094919590481107968,1095169656358227456,1095351526750484864,1095872381727479936,1096117955074105088,1101763324511186304,1102382143399269632,1102843766484141696,1129832000941901056,1131229995617244800,1132614349476009728,1132966468074614272,1133810794221801856,1134097899899377536,1143693887630730240,1145020139172129536,1145602571097053952,1146255612287729280,1153282110061189120,1153395016166202624,1153544794556784384,1154089117236952960,1154209685558171520,1154456216681249536,1154943403412027520,1155054660244121600,1156006287558141056,1157317008497672320,1158347014670132864,1158864021358233728,1159061967810978432,1159705388271330176,1160021841461912448,1160054281350128384,1160300056558791168,1160308882716770688,1160623343042019968,1160759923002231936,1160939452634666496,1161215296909017728,1161820298887863424,1161933582944846336,1162536355835288320,1163809177983446272,1163912467652160896,1164185283975493120,1164767677244452096,1165354855109711232,1165926910393567872,1165926910393568384,1166063073741131520,1166330259362301440,1166936944966743168,1167866620703225984,1168173070914786816,1169027631967046784,1169663527645706752,1170279945646967424,1170811039827446528,1170894503927889024,1171041215714699904,1171211743096194560,1171581179003421312,1171726623775534080,1172289608089050496,1173230210222240640,1173793121520878464,1174070610767977728,1174996090320432000,1175204104176381696,1175242896321238400,1176312965948448512,1176378520034381952,1176717792385803136,1176784239825009280,1176899898999183360,1179764607826002688,1181607110141081984,1182083542272959232,1182318120502368256,1183271186629536512,1183473535423719296,1183956427187401344,1186165582270482688,1186502513864638464,1186846832802736384,1188277434870033024,1189269297437549056,1191742717564070528,1193808772927091712,1194922711349401856,1196295211098415616,1197139189352207232,1197240000824171648,1197720457346241152,1198427031003111552,1198632983274798336,1198984345958836864,1199192840852373376,1199686173677816576,1199771832505966208,1200156524136106624,1200416902234462464,1200578152485877376,1201115779312352256,1201304517354978688,1202552914026910976,1202662482934564736,1202662487232814848,1202826348825240832,1203132627235232256,1203637376090060928,1204051067339635072,1204095391402194944,1204316668115496704,1204450911613674624,1205467268384479872,1206198924646506240,1206715626392179072,1207403993686544256,1207804112839827712,1209845764198083584,1211862818279517824,1212348119518459392,1214468115376082560,1215384313505335424,1215408610135114112,1216212932955510528,1216353051968362112,1216861541736356224,1216863676336891520,1218051664291152000,1218104543928648832,1218800393053400576,1219657707181797632,1219699145026398848,1219957873855946240,1220062430539961344,1221075080749770752,1222150059528734464,1222254066452610176,1222790795628242048,1222865601075380480,1223091649497003264,1223151194923807872,1223420094236095872,1223443630657641728,1225189552042779904,1225595203113949440,1226206806457202432,1226246251436497152,1226307755368045824,1228266814506156928,1229916112012470528,1229959542721326080,1230398076062097792,1230575131794474752,1230679417895574784,1230761395935567744,1231038060549269120,1231446224176456064,1231503188327196672,1231505456069951616,1232045934759720192,1232908123669385216,1233331229487768448,1233598174590839296,1233762993960131456,1233764059112037632,1237419351158798336,1238026522095817728,1238352149340737536,1238445745263425536,1238591671072637184,1238610809446526464,1239213243034520192,1239963487922089600,1240302992201816192,1240475993483896832,1240776435036602368,1241188721832696704,1241517196641669760,1241561658137743232,1241754244470666496,1242573758590597120,1244753299874351488,1245468295966061952,1246230047071154944,1247558738152408576,1250704440919829888,1251229182844030848,1252527607292758144,1253353890280943104,1253540837322222336,1254241707265648768,1256674514180403200,1256983580027464576,1257631879571104640,1258206340036418560,1258934014870979712,1259128246176488576,1259235139323302656,1260450030951612416,1260585202163097728,1260898494256935040,1261253842671908096,1261421999231535616,1261705673230699008,1261751715280478848,1262123144052263040,1263732043096088320,1264639994887264512,1264673637366153728,1265985590961480192,1266942548329434112,1267289165075531008,1267355926046747520,1269138715432182784,1269138719727661568,1269638134229067392,1269774649765271552,1270140821499608832,1271096675060378624,1271649969930799872,1272088876928180608,1273088783971336576,1273307891725715584,1273456463234876288,1273685372108354176,1274247291269220864,1275492999287723392,1275760219272654080,1275981324189764608,1276054682231244160,1276297674297720576,1276688069644366592,1277232907719022464,1277756687570196096,1278042010837800192,1278736391086871168,1278756555956826624,1280674894509973760,1281026635151594752,1281225032575659520,1281410781322153216,1281584774741482880,1281810110201241344,1282030970304135296,1283540844582413952,1284194603029215104,1284932139108595712,1285142833024133760,1285224879783795968,1286399437381641728,1288658006059681408,1289020673097509760,1289211434068336512,1289485349902909440,1289812355829447040,1289986903300047744,1290050262657404544,1290081323859442048,1291640053390590720,1291844081516924672,1292166062330701056,1293045504129952768,1294193561772534016,1294793345366747776,1295880762365052160,1296259647200187136,1297161594628972672,1299122058220640512,1299314816353108992,1299436415467829888,1299827699870774272,1300212116623817728,1300334299853917184,1300356053864952064,1300727345195414272,1302697738753921536,1303034120592956160,1304081783374935680,1304274094830734720,1304733106575117056,1305209190813293440,1306197930941817984,1306242834823730176,1306917144688318976,1307226283551364224,1308753230324383488,1308772574857190144,1310256365798876032,1311660540930405632,1311859655613503232,1313265900922425856,1313405848136604672,1314045729544380288,1315371255234720896,1315480072526858624,1316238426312036096,1316376453675730560,1316607896578157824,1317275544951049472,1317473353964505600,1317956898562626176,1318204460477280512,1319616435270604800,1319952542230868096,1320147671184560896,1321738565727229056,1322022514605143680,1322272099449344384,1322796261553165824,1323922779935068160,1324316263362759552,1324361343339619968,1325292217371985024,1325705942981586176,1325899461324069376,1326398777041821568,1327436471204300288,1329697032751848448,1330758920465540224,1331106782752978688,1331208070965684224,1331789850056080640,1331894033078063488,1333697880686680960,1334374153353733888,1334638379741833728,1334770870892758912,1335071381164641280,1335522692031527936,1336402404413235840,1336712848946493952,1338388813904333952,1338455643596995072,1339274053906752896,1341599143042727680,1342008814203085440,1345041748308955648,1346164491417165696,1346282375381981952,1346515918524100992,1346883876962000000,1348007784003797376,1348795004265872000,1349256249394224128,1349726324974648192,1350007967453813632,1351486711809893888,1351956512512484480,1352571723628352512,1352692734330406912,1352743419240352384,1352879238991076352,1353302001211658368,1353355434900703616,1354258473248294912,1354610798005917312,1354669553158591232,1355499104617153664,1355979969155873792,1356633384004567168,1356850743709013504,1356856413065960960,1358109963696352256,1358301480583401728,1358325021299899136,1361501403017141376,1363721110836082176,1367469041032593024,1367551813643361280,1367876852471768320,1368236912466084352,1368634347263005312,1369626141111655680,1369772925913456768,1370654257498865536,1371714251131163264,1371753799190180992,1371872413302877568,1372458109403442432,1373569711363357056,1374151146855898240,1374676889508967680,1376326912864142336,1378546345804178560,1380443823700583552,1380553946662309632,1381245956088755200,1383292181587562496,1383598326856370048,1384866854036828032,1384938730313376512,1384982332821602560,1385719147346936064,1385742821206707456,1386691867244096000,1386704545987569280,1387477678757569024,1387655524762487552,1388278982215216256,1389687246155888512,1390295486540598656,1390852870215428608,1391331222197901696,1392065975139109504,1392443137691842048,1392973720770987392,1393165997866957184,1393328553789078784,1394045267867814400,1394479501945576064,1395759333479862016,1397232271040309248,1397898884325362304,1398051402907424512,1400157173832960384,1400201326096359808,1400691059743459584,1401010605309570816,1401717243394126464,1402600323029833984,1402721307964211072,1403348682426861568,1403545284553914880,1403650734590873856,1404831472640252928,1405848383457312512,1406792623427291136,1407138625992348928,1408135749896103936,1408135749896104192,1408863003822944000,1409166984427845632,1410031887762012800,1410448259071414528,1411226507145210624,1411292580922301312,1411492795117800448,1411650884274568320,1411867767238390912,1411873672817879936,1412116767967011712,1412126319974269696,1412228093519039744,1413145468469632000,1415935371854737408,1416764167393932160,1417200020676904704,1417404048803018752,1419662338310872320,1420687362321632768,1422012892308493568,1423078520938173056,1423481972986629376,1423486611551423616,1425130935485979520,1425236724824667520,1425374610454574464,1425397189098547200,1425909733315616000,1427746463194276736,1428562506980546688,1428781653392160128,1429284061486392576,1429618420396285952,1429800419634352000,1430007784951049216,1432420624562923520,1433166540130924544,1435716719905089280,1435776922462177408,1435887934481462784,1435977686413453056,1436196450572176896,1438252232373823360,1438490650303065984,1438843215578245888,1439084008624844928,1439283711720415872,1441044132915458176,1441520393248858240,1441623777405642368,1443266623871074816,1444161214019904768,1444528416543875456,1444926572896853376,1447511765251613184,1447761801067600128,1447925976193082368,1449073419655962496,1450096442210453632,1450153273218061696,1451503404777191936,1451544876981617920,1452337006389928960,1453676525085505408,1454347089739329152,1455070083058636672,1455502843963911296,1456903763514134272,1456907504427822464,1456920737222542208,1458024268941320064,1458210842321624448,1459546263999675264,1459590725116218752,1460293317343353344,1460807304669403136,1461469893567499904,1461916982483364864,1462641629366243712,1464134289414972288,1464773616771051136,1464837457164974976,1465871650930738048,1466070426313113088,1466654606288277632,1467011054214315776,1467956668578931840,1468305213760240384,1468882118064381696,1469189191046042112,1470240530320055680,1470849350522703488,1470926586919626880,1471788161655374080,1471998511973713152,1472029470098019072,1472094341284011904,1472114647889153920,1472382752631394048,1472401586064123904,1472481502519806848,1472500473391093120,1472821908743764096,1474090607723007104,1474929672534527360,1476676792214925056,1477612029933470080,1478514389681253248,1479297757353952640,1480565666057619072,1480602018663176960,1481311753417900160,1481667548507617920,1481854770426696704,1482922018261625728,1483371791532150784,1483427385588999936,1483683911099065088,1484845034034212224,1485197354494658176,1485449383176277248,1486484367214724992,1486537113709628672,1486801267085903488,1487362258530962944,1488064949540111104,1488528982101264000,1488715482466110080,1488904946359359488,1489504004102020864,1491559506730816384,1492278518615941504,1492836834300292992,1492944375984949504,1493367245581725184,1493842161589005440,1493934284343845504,1494157691363079168,1494432053873688448,1494485758144731520,1494564162771398016,1495325853747745536,1495340933377121792,1497899703093177728,1498122491636872064,1498126030689316736,1498447607777405312,1499209530679741184,1499290379143445760,1500140065408640512,1500607765872799616,1501012287368950528,1501853065870482304,1502063317405058304,1502284452385828864,1502616539256347136,1503091085900019712,1503124243047563008,1503203059990907648,1503766009943842048,1504365454235966848,1504443450842002176,1504453896202493568,1504563057091834496,1505825635741455872,1506282933795985280,1506805751574450048,1507571286545131648,1508390899448035712,1508482266288333952,1508988758895110656,1509745124111150848,1509842495314395264,1512752769450543488,1512865022715710464,1514641897929708800,1514736047908364928,1514812871987565568,1515219691289797632,1515693894335906688,1515869884914637824,1517211697121965440,1517239773324267904,1517475790365881856,1517997504338290432,1518352719609121280,1518401299983488256,1518753178064521728,1518943638389520384,1519362376225775744,1521599470071180672,1523116177642231808,1523467402887806336,1523586253222618112,1523861990123367808,1524205557443256192,1525359868557998336,1527224404056238848,1528190393741232000,1528722694807934720,1529609897612606592,1529995065983144064,1531097433767946240,1531580424311162880,1532708866839099648,1533299819978845312,1533421728330980096,1533514911941019008,1533568375694548480,1533950318546008448,1534719152051775616,1534836250044023168,1536034236678159232,1537203704733382016,1537367257087358592,1537794524729363712,1538365922883198336,1538625407627245056,1540371883066128768,1540651193376855680,1541286711100812160,1541515856194566016,1541891816154999168,1542850487215495168,1542901000329134336,1542933401563275136,1543356369938446720,1543370904111505408,1544353553970169728,1544707665434144640,1545050717357814272,1546270625515781248,1546780420950246912,1548117972546002560,1549208138324951936,1549228032613591040,1550208900064886656,1550492952021749248,1550602907481860480,1551096996222472832,1551279240274526208,1551346520939849728,1551402252435630592,1551761857160285312,1551843775071158528,1551929124663349632,1552504684640838656,1553656423070589568,1554850591480752384,1555410036744248832,1555883930548870400,1556005701461744640,1556162240133919744,1556577584947813376,1557524191442926336,1558872261424381184,1558885180679234304,1559416798846597632,1561389253988468224,1561941170170300288,1563260687202320256,1563853976804546688,1564321200527368704,1565012935076040832,1565832689713351808,1566012013187728768,1566020598826667904,1566530913957066240,1566553007268519168,1568436951724556416,1570111709795010816,1570659781981537024,1570828286435383424,1572216243769902336,1572454940873010176,1572601725674655488,1572762739705066880,1573089397737635712,1573121554156399872,1573358945589364608,1574781163879398912,1574853491129437440,1575107070293783296,1575509800786419712,1576088036528753408,1577076089460578816,1577838944371859456,1577900791900994688,1578256136019422080,1578585611551014528,1578748824604827648,1578748858964566400,1579147088331814144,1579314622119638400,1581002544266699392,1582663189077609088,1582945145090883200,1582991152780947712,1583023618437189504,1584734595969095424,1585063422960992256,1586427298416103296,1586838374030594048,1587150257375155584,1587405000476135168,1587611884756030720,1588269594571752704,1588627348168467200,1589209333416214272,1590192369827025152,1590462948470580480,1591250718488330112,1592142284979684736,1592242645479324288,1592629707931918720,1593473346884952448,1593599855144838912,1594422496000994048,1596808642391186560,1596812426258489600,1596840493868996864,1597468108851256192,1599685347062685184,1599832131864165760,1599832161929255424,1600035129199827968,1600882268549215616,1601489813146914176,1602106540386957184,1602720342752926976,1603202920982243328,1603370081109395200,1603796073145515136,1604029444488823552,1604147165248626048,1604200899583092992,1604224470364124416,1604422214954487168,1605951944865784960,1606475479904001792,1607546129351189888,1607738921843325440,1608061254844374784,1608497864040134016,1608747311444717440,1609392862209121664,1609479792346820736,1610657197502002048,1612301860739133952,1612339420228653440,1613573690750097024,1613862480054872320,1616103353471994752,1616879612977156352,1618920714579089280,1619166249269416832,1621523537776065280,1622697949337226112,1622769245794819968,1622881670858925312,1623911878894401024,1624733561973468928,1626878090683867136,1627064973301021056,1627688941853399680,1627811571760152960,1628349890076998912,1628379675674825984,1629354495811136384,1631186458277453440,1631578537252535040,1631669453120152192,1631796309274519040,1631815069691601792,1632041328568823424,1632737005895324672,1633045217045425024,1633504675468010624,1633840370109794432,1633890943348374528,1635687790163070976,1637642442663523712,1638448388981845888,1639910911246160896,1640335009200444800,1640907889118460672,1641326979142898048,1641386833807056640,1645204475617697536,1652043330567438336,1655450579663164544,1657597483491271552,1658578797618935680,1658818319354849536,1659211974581862912,1660937971614743808,1661023419988816256,1662778377985657344,1662871286717909632,1663262163101591936,1663349849153677952,1663839681584291072,1663956268471724288,1664401914277704192,1666261218505739008,1666378698746614528,1666805889078252928,1668302977238052992,1668331122160137472,1671329451713079808,1671824128866481792,1675399225287958272,1675699568051964800,1675850407304241536,1677716789636812160,1679365202380970880,1679458897091659008,1680452683804651520,1680532535836237440,1680980518105612672,1682022481467013504,1682144424177509888,1682417167486191360,1683330453627242752,1684701849569650432,1703379704562897280,1704827791440274560,1704918191912744192,1705586493119219712,1705751041906319872,1706149129539465216,1706270664229782528,1709423887483553664,1709424712117232640,1729779631580923520,1730206718833295872,1731520703948128384,1731752185504899200,1732176768792860032,1732988758129091200,1733782571164433408,1734159875452345472,1734682143474825472,1735171735387308160,1736329589850340736,1736524821883784320,1736555475066523008,1737895053891363200,1737897841326227072,1738863620555712512,1738915641198942080,1739397914486670208,1741417790361772160,1741515470802472576,1742342784582615936,1744001432233632256,1744017272074321920,1745011677261492608,1746315255670491904,1747132467687929600,1748816983925915776,1748906078727268224,1750087538330587648,1750305241632853376,1750460792464604672,1750545141328217984,1750703234778677760,1752188773770447872,1756261467922806784,1762246625107355264,1762899533150392704,1762951588152369408,1762951588154154240,1763010888766991104,1764481588648685440,1765847182089067008,1765881576187554432,1766164322474245376,1766620929036759808,1766702297192943488,1767957488499891968,1768730586908531712,1769541820331697792,1769853321424222080,1771018842404904832,1772335163982155136,1773233087024464768,1773934438004155136,1776027972567260416,1777269424274555776,1778836056545324160,1778839767397091200,1779145878305570944,1780017206911516928,1780464093964396672,1780497934010525312,1781605382738862592,1782874287875828864,1782917757240791680,1783463523029420544,1785113030990638080,1785260090670874240,1785659488269249408,1786065654735928064,1786103278648516352,1786138222502901376,1786444230333492480,1790303791024282880,1792207664128299776,1792830060723673472,1792948357008256512,1793120808535107072,1793628851625008512,1794118516552814336,1794423901612865024,1795037875776233984,1795467063268699264,1795603162191997824,1796837467073673728,1797472370615364992,1798237802506593792,1798927947915336448,1799336279047495424,1799385928868168960,1799515812976881152,1799761927489297152,1799850987927155456,1799905929149972352,1800070958971784960,1801199745097109632,1801892849742096128,1804324694582692736,1804324767597400576,1804933033754317568,1810475401285228928,1839493604790492032,1840865211187303424,1842543963578582016,1843669279369093888,1845354620239612928,1847118992806013568,1847190701579945856,1847460116290325888,1856027854494006784,1856123481943041152,1856522497282307584,1856608430990652672,1857241028137065984,1857362833406564608,1860318973495642240,1861681749446432128,1864205575306773888,1873653644358070400,1874330118886582016,1874375473741569664,1874951106733026048,1874954641491354624,1874954645786146304,1875301369907249024,1875431077919202176,1875613386395668864,1875741857456294912,1876069099605221760,1878189370339859328,1878953187324368000,1879147564661258240,1880428800652241664,1880503434295761024,1880527279954062848,1881611050526390400,1881612734154248320,1882264022993430656,1882626964910248960,1883032242321525248,1884925429543282048,1885744595770968832,1885744600065811840,1886463401496969216,1886717045085837056,1887302260150228224,1888086452459286144,1888836972229230976,1890126871163145728,1890216962396479488,1890542971890672896,1890612511706016128,1892991957946311936,1892992267183979776,1893517696302461184,1894563160062140416,1894996161487458048,1895965724582481280,1896214179850241408,1896636873351587840,1897310564741680512,1898416368496688000,1898485156696053504,1898875311527105152,1900545847646195840,1901132845117712256,1901175799087508608,1902873513759656576,1904399047486667392,1907427720985244928,1908476521936298496,1908964185404897152,1909045549271308928,1910053698353680256,1911410083380807936,1911871087990571008,1912118310604657408,1913048360003157504,1914595677804443648,1914644846590660992,1919266093961824128,1921255247935490944,1921293254101835776,1921344205298938112,1922778419431649280,1925701917774697344,1939308236730745472,1942951949545191168,1943168617756289792,1943422948539962240,1944042802521121024,1956946052506753280,1958202240244615552,1958785020063453312,1973393612591331584,1974579637737581568,1977776953135753856,1977906588137089024,1978812418219788288,1991357777161163904,1995565191542364672,1995699675562032768,1996825098731349120,1996831828951826176,1999250788878655360,1999411936053433856,2009135638930614912,2010655404533609216,2014749990114568320,2015603344284544768,2017135002642810112,2052569921645067008,2053953008490747392,2055059112897283712,2056895194225684864,2056895194225688064,2063293355469201152,2063705294371650432,2063993125898602368,2064838031864808832,2065903359853755392,2066035777984385664,2066035777984385792,2066251179185657856,2067343097310013056,2068709755916919552,2069588300054515200,2098318989863211648,2099533361800789632,2100934448852549760,2103614787618232192,2103645230346699776,2109852523240471936,2110551602180192256,2111294734600515072,2113961462615081472,2114136972159561344,2114811453822316160,2115952197141317888,2116551740212362624,2122880116823483776,2123400563781032320,2123432247756223488,2160181598552871040,2160210705546460032,2161573442833511040,2163226700308494080,2171179669768878080,2172420060626469248,2176116580055936512,2187254083088520192,2187881835511177216,2189957816542364800,2205619397609830656,2208124536065383424,2208530698238308736,2208752769527485568,2210319539237234816,2214973561500636544,2214973771957744128,2224320853443995136,2224491655704638848,2225982838287401984,2226267607501444608,2227204052104933248,2227233773279444736,2227233773282607488,2228817825940263808,2236381950543626240,2236900335916533248,2237377042922610304,2237402121237340032,2237670711312427264,2238057533249596800,2240605346504273280,2241695439264955264,2252512954353689344,2253826832091026560,2256345954965239680,2256410856215182464,2256422164865057536,2256681615250558592,2256687898786090112,2273004926219966976,2274076297221555968,2274076301516712704,2290479876836755840,2290574400476477952,2291529055743487872,2292799369631237760,2292861388958880640,2349916559152267008,2350826847404607616,2353173926773144192,2353801065013205504,2364195122092662912,2364587647743373056,2364836107306975872,2368814823275272064,2377801239183645696,2389966854309408512,2413565565001399680,2422442334689173376,2423337955632676096,2423499381975837056,2424913628807069056,2424915789174828160,2425650847058388224,2427195557815626880,2428145295344110080,2429183303040388992,2429268309033277184,2429392661221943040,2435433202010203264,2435696569404865536,2436043980719301504,2436346208978391936,2436987189897259264,2437760730687096192,2437933525811466112,2439718930832024960,2441483303397892608,2443419990050464128,2444134496514446976,2444446482939165824,2444731325169827200,2445187691214892416,2446993162322393088,2447183820215313024,2447341604429683200,2447374070086854528,2447815253423324544,2447889401738675072,2448308212589610240,2448933731627261824,2449594087142467712,2450558049601994496,2465343629137442816,2466730839150018944,2466774368643278848,2467788122659245696,2469900005324049280,2469900009618822656,2470416951882130560,2471495809011745792,2471521269578009984,2472368993042955520,2473296504116465408,2473754897386327808,2473843786029439104,2475572527545073792,2476337856358290560,2476711101899811456,2478759973099060480,2479327870854949248,2480523216087975040,2480799056067605760,2482539166362050304,2482721857094021760,2482813425794003200,2482813430089468160,2483288620975449984,2485211533668928640,2485921779525500288,2486189510606565760,2486388560866377856,2487010845792466432,2487100490353908864,2487514387758522368,2487647359945858816,2487975808980333184,2488226252817987584,2488960249844340352,2488974302977323008,2489275328645218560,2489533370280291584,2489730281645658112,2490455242060072064,2490455242060751360,2490494240363793792,2490695859013226240,2490975272405858048,2491770464126012032,2492027264515309696,2493698663923670144,2494399362068211328,2494836001327878400,2494852601377178112,2495751967528809216,2497895053130247040,2501224305619721984,2502097283492466560,2503373296800774784,2504879597666546048,2505945535535013376,2510891108771852416,2511020022215011072,2511623211717232640,2512346415490500864,2512420696949788800,2512446810350495488,2514364118112563072,2515570454165603456,2516322146457318144,2516606022320239104,2517080495947872896,2517705950560296448,2518093214875967360,2518822538978075904,2519592124102911872,2520280250878310272,2521336396221143168,2522341212409740288,2522906262603157760,2523835929748697088,2523840155996531456,2524394000619171840,2524541476912991232,2525794778432811264,2526450362241718400,2527045889522365824,2528877194857837184,2529337507976700928,2530119196321115776,2530509724810602496,2530629365419780864,2530820027607910016,2531326283993100416,2532236546477272192,2532554129243495168,2533306985471073920,2536332154276141696,2537908647792027392,2537913801752406400,2538281386529032192,2538286403050847232,2539347255677275264,2540716216374137984,2540822937721559808,2541126609088680448,2541549062071571968,2541707185587555584,2542192757410302720,2543230185287345664,2544456862306151680,2544472457331853440,2545305410405677312,2545505281002947200,2545590733671247744,2546887573277984256,2548152217808246528,2549282305307437184,2550856260497950464,2551087359802740096,2551258990991085184,2552160826748508672,2552643413569299968,2552928187080872832,2555356080553959168,2555403428272657792,2555604467102461440,2555713215673881856,2556605469360226048,2557815649409852544,2558010366047159424,2558321974514561408,2558649835138475904,2558736322894795648,2562026950744670848,2563515036653763456,2564424130905288192,2564860667085575552,2565109843908256896,2567842095943560448,2568552487829583616,2568588664341691520,2569277680172652928,2569708035895872768,2571255053050563072,2571671561799507968,2572388340301051136,2572713104253752448,2574163909846061568,2574640827310228992,2574651852491249024,2574761734934286848,2574848772446200960,2575509780797922176,2576459518326275072,2576622898882026624,2577467770489127296,2579266674591294464,2579280762084084736,2581058706745690752,2581380077673636608,2581380077674117120,2582416710980577792,2583211103836375296,2583248384152322048,2583365245917474816,2583651904920127232,2583851917252389888,2585189473147457408,2585711397573234816,2586182156053386240,2587268134239449728,2587491678697364224,2587993017344962688,2588874825669925504,2589614865714845312,2589844972883309568,2590483896512845568,2590924250920129920,2591091789004351104,2591754107321120896,2593884960855727872,2605271846869877632,2606779964507034240,2606979835105147520,2607380156121387648,2608247533357159424,2610488514148351360,2611836167511100800,2615280147167407104,2615968858057952640,2619561508006403712,2620378823103824256,2621085018806417280,2621138310760174592,2621565333592122496,2622175287664315392,2622700373185644800,2624654072204568448,2625405206150216960,2625424962999379840,2625527084437039744,2625541412447911424,2625620710428582272,2625820512307604352,2626108893591397248,2626677723354947968,2627035300857269632,2629099187262355712,2629899631727265280,2629899631727265920,2630076966632012160,2631652528139944832,2632243897891836160,2632899099449032576,2633120655335891968,2634608741244388096,2634608741244966016,2636318649330041216,2636996807485877248,2637042196700093056,2637741490390750080,2638053472519840256,2638553754605793408,2639615638024336128,2641576685735609472,2643629405222112256,2645295921252503424,2645295955612242688,2645449715440904960,2645509810623287552,2647710311349589632,2648753232486356096,2650584125507949056,2650863985573885568,2651072927142789632,2651439923508503552,2651505207011569152,2652547440955473024,2652566063933448320,2654170736729886976,2654423998066862464,2655547630230478464,2656454216222569216,2656959231361709056,2657122444414522368,2657451056656635648,2659860636389069312,2661252033994618496,2662208372887759744,2663502184540697856,2665304249738571392,2665414200901834112,2665842293176679808,2666306798185155200,2667464656943675392,2668055743228735232,2668196480715605632,2668524444418987136,2670770991487735424,2670773907771422720,2671209215590782208,2671550540937739008,2671606959627000576,2673699059671833984,2674134771219567616,2676566272464334720,2676589259128900736,2677157569201482624,2677191860220980480,2677321641247692032,2677613351131443328,2677851743291189888,2679806537525967616,2681243457490130304,2682094616928734976,2683210316288400768,2683345934175922176,2683452758602312192,2684594390974873728,2684806459279262976,2685318075783946368,2686607906002083328,2687359284761589760,2687733913283870336,2687768066863960576,2688986325747403264,2689129601561746560,2689496872803968768,2689915958535673472,2690065251596080384,2690697646876721152,2690763205257368832,2690830511689102336,2691890685712023552,2693095097621419648,2693940725141960192,2694329570005647488,2696628687474414208,2697238362376404224,2697327113581223936,2698026093739557120,2700725635303873024,2702527013306504064,2702619273499324416,2704122988794215168,2704955971931335168,2705188999676567936,2706795626682842752,2707280545670705792,2707587378134583168,2708540070600397568,2711092621203766912,2711324446359728384,2712093451662656256,2712240064671438720,2712384409927302784,2712583146653885824,2713617271700011008,2713932247421725184,2714269007216940800,2714977745540449664,2715786505062211840,2717510195697147264,2718032566799805952,2718118053828251648,2718222954109355520,2718818202217242240,2719012579552367488,2719852224183359360,2719908367995903744,2722195455261832448,2722273245708777984,2722715043225172096,2723350492226010880,2724895585236306560,2726437611640855680,2727596187657230592,2727700950499268864,2729211164080363264,2729503359295054336,2730508416002618752,2730989658498251776,2731772377633104128,2731866347221858432,2731922422315395968,2732515029018125312,2732864192679430656,2733055335904034432,2734163982926890624,2734319087081051264,2734807643906462464,2734850353060588928,2735006689870184832,2735638840336737408,2735802976806568960,2736054627915080448,2736312982698161536,2737611093613805568,2737737292637935232,2737918849495239936,2737921155893258496,2738626591386423424,2739231150982741760,2739624711723101312,2739975142397573120,2741440172922171008,2742201172407646208,2742615963169132800,2742718080312489088,2742789930821144320,2743812644136889600,2744932943406490112,2745244002118134400,2746037712074342784,2746394056920864256,2746592591785639680,2746843589674667264,2747384888699406080,2747843836017055488,2749042299395966976,2751173530888036224,2751749709340268416,2751750087297310592,2755569515814791808,2755903045794863872,2756665763266917888,2756685760634707584,2758938385082121216,2759049470116155520,2759340943776878720,2759588063311504768,2760011096114471040,2761104629147542400,2761401084970307712,2761731595589115520,2762071409106142080,2762076666146759040,2762356594934263552,2762617557147396352,2762779322795219584,2763917802663393920,2764723912188128896,2766386648647246080,2766498012855959424,2767087346792302720,2767419850275077504,2768116146078155648,2768190500551905408,2768919442402016896,2770050431846042624,2770647462363429376,2771071908211662464,2774460229385992320,2774568290763823872,2776464261127797248,2776562598698116992,2777015593193689216,2777098984278522752,2777751372630874112,2778069062775645184,2779284538516313600,2780050795041827968,2780434524599787136,2780524585769652736,2780622648462300160,2781012734572656640,2781309637071878912,2781770637386074112,2782037303315690752,2783309850585780352,2784262989728165888,2784281750145869696,2784445680457082368,2784832914708609536,2785252515833393664,2785927684692570496,2787162401891160832,2787507098786135552,2787859938939149696,2787939447373738880,2788048780061707520,2788992130973584640,2789398778477164672,2789405753503977472,2789721038463275520,2790417540424293120,2790884008232725632,2791363193439212160,2792200501607971840,2792315366213367296,2792477995151179520,2792919650932258560,2794904441219064448,2795833567198957568,2796074252871563776,2796224099985984768,2796378817592437248,2797216031272977280,2797842095770634752,2799809779202757376,2799839702239434240,2800103825548199168,2801525356645366016,2803442840498734720,2806027925479257216,2807036078858147584,2807462655009880320,2807933447849924224,2808795537980708736,2809470496386299264,2809521108280956800,2810585920868186240,2810953668852136064,2810974525213376000,2811321837744375936,2811363550466857344,2811484217572663168,2811882073278282496,2812250821990695936,2812412377185443840,2813020961166816512,2814629409239942272,2814942392095501952,2815276162594025216,2815415285174565248,2816082757452460416,2817075272854044416,2817973569559904640,2818456736203265792,2818683750991979136,2818727009902345600,2818957013992481280,2819079300299524480,2820519145136386688,2820929640930943488,2821159713738542848,2822330113802737408,2822335370842658304,2823677664085658752,2825024531473340416,2825325213544431360,2825857725062378624,2826254713186397440,2826770319713589888,2828248471361734528,2828968376599542144,2830734604590471424,2831562468126593024,2831697712353381376,2831963931606829312,2832497985019340416,2833291244003645440,2833496852677924480,2834500058254189440,2834595406528921984,2835470068028446976,2836609093355562496,2836686398471189120,2837909055400871424,2838537563735201664,2838876114534014848,2839231634746334848,2839231634746334976,2839397832801325952,2841283151645302272,2841679456867414400,2841680934336158976,2841680934336159104,2842112183412398336,2842282088022707968,2842312874347797632,2842462137347560320,2842650153836732928,2842710382160936576,2843077206728009472,2843077211025217408,2843086827454803712,2843695170919181696,2844308831549635840,2844933221011789952,2845059871007824768,2845218578639288576,2845358143600048768,2845408514976562688,2845447204042284032,2845665594538512768,2848334010475644288,2850426479892014976,2853360530961651584,2853372247632375040,2853763295814123264,2854432108121022464,2854727528856212992,2855386170682263424,2855790657816672000,2855799415254881408,2856067008897130112,2856257606661059456,2857547329505058176,2858325611939289472,2858896086675180928,2859908324567852416,2860951829821782016,2861037080628898816,2861367067259811840,2861452348130844160,2861792754354276352,2862337802883401856,2862873192032202112,2863526233218817024,2865060636057386112,2865535629374939520,2866228901519900544,2867032958053059200,2867052199506324736,2867203584218146944,2867729563092848512,2868367245477025920,2869130517001766400,2869685285040721536,2870129010998688512,2871162547632415744,2871730307948650368,2872373629627611520,2872712618509070080,2872720967925458560,2872974856329379968,2874216647336589568,2874425180884785920,2874829247112790528,2875903332533220992,2876148729784835584,2876148734080688256,2876705018245556480,2876822146296910976,2877080497170502144,2878163172526469376,2878546249251932928,2878572534449316096,2878797526311723776,2881215970856879616,2881240087097750528,2927186036623328256,2935174641435947008,2935224527473059200,2935415125246460032,2935446392608812032,2935589908933376896,2935654853138398464,2935715150178454528,2940003108087120640,2940050734981714432,3016329285438157312,3026097243660272768,3026376377878137088,3030301501255870720,3030820432081929088,3030893377804110080,3032020450247993600,3033570727283474304,3034021728905651328,3036818783407270528,3037368539222530944,3039671225803688320,3040162810580901376,3042344245950391040,3063650376111451136,3063960605892490368,3064151577318715904,3065135949463337984,3065513047592503552,3070143267150880384,3070832695300382464,3072348715677121280,3073014263809462528,3073747187092773632,3074211494534301696,3074640299772139008,3075067700558242176,3075976794811325952,3076010742232557952,3078733884642456064,3079104660578372480,3079468598933810176,3079642326059886208,3080205580955810432,3080411881119844736,3082984910130769536,3083265904070625792,3083384234712571264,3083765357227100800,3083915298828722304,3084195266274202752,3084427916062349440,3086397068370475392,3087965251128848384,3090505359208876288,3091044532222339584,3091528115477228288,3093722358432322176,3094852140989111680,3095326687633750016,3097756127294671488,3118420619409587712,3118424948738692352,3119203643485532416,3120241513743098496,3121223102749655808,3121385658671190784,3121684764489788288,3122380407457501824,3126624316184360576,3128834506352463104,3129586606667254016,3144237908341731712,3145370645837100160,3145382126287048832,3145766676180517760,3145802036649641472,3146188235810977920,3146523896095375744,3146576878812071168,3148732406937031936,3150099301752709504,3150770626615542784,3150999325035676288,3151407827963355136,3151443115414582016,3160815489969826944,3164518640833807872,3165528615280166144,3165713981772896000,3165908389171910016,3166841329084630784,3167640566662056960,3169614331770315392,3187950092615927552,3188530845210403840,3190645098694237184,3193194629937678464,3197019139399144320,3197508598168785408,3197515087862709888,3198154591313378304,3199430372695297792,3199980330371548800,3200232157189930880,3201198451816207104,3201594074140023936,3203410359973647360,3203444071173553536,3204325050568185344,3204665315057342336,3208758831207224704,3209150124204756224,3209308900555979776,3209958746287926528,3211272417868507264,3211784863301909504,3213304903767768832,3214675028399417856,3216947242193857024,3217172912659449600,3218697767783768320,3219198487955502720,3219428015303785856,3220964376644865024,3220982720449432448,3221458469681100800,3221735473597064960,3221744063532438400,3222024473357675136,3222138444609365248,3222969370457259648,3223564476830329728,3224908977688888064,3226519762223501696,3226718494654224256,3227326352785075072,3227446822324601472,3228023859770327424,3228730738371591040,3229458030954057216,3229556024930689024,3229873238328746112,3230171064244273024,3231500000140463360,3231702860037325824,3234395078681331840,3235993150111995392,3236259854698180480,3243520063118545408,3244162250924266880,3246152023374303104,3247469062209123072,3247469130928600960,3251244858154433536,3251372053609078400,3254081490778611584,3254620736808973184,3254857814706920064,3254930932230273664,3256598792586547584,3262674487682440448,3262723514733841792,3264828357946121984,3265311627666274944,3266534357610950912,3266576684513815168,3266873724451739776,3267081566509086592,3268703483599047296,3273081120426349568,3274761620869767168,3275218193073023360,3275979398716956160,3276355500412926336,3276414466021403264,3277594512580808448,3278155091710087040,3278602386784473472,3278914205706332928,3285329611240097920,3286699293490332672,3287289048334694144,3287489258235171328,3292685210887016192,3293451020735654272,3293635012838798464,3293912089768428160,3294248609046258048,3295332796231463168,3297249936488848384,3299044678766326912,3303137301563100544,3303258492657447296,3305474760206403200,3305539631390995200,3305693425580086912,3311813685256807040,3313742056853382144,3313861186362477824,3314967982254801024,3315967953720905216,3318463707678938368,3318612893365741824,3318722672728544384,3319426463252487424,3321431014682434176,3324963367588643968,3326650224581677312,3327647271174824704,3332761855668238464,3333830379107891968,3354222505874657280,3385389606068550144,3386162214851684864,3392181976588782336,3392962110153133056,3394087945636978432,3397839440654502656,3400048535611299456,3400851484041556992,3400852617912637824,3410496365678521216,3410702768924111872,3411147173483832320,3411282585212210048,3412625024486012032,3418592726925673728,3418741367154043648,3419918875386344576,3420019270246122112,3421462898950860928,3422189774920460928,3424863550977729536,3425920250370248832,3440684419492448128,3444723234938138368,3444888196042856192,3445742172986796544,3450734539887203072,3451263611151284480,3452101198480935680,3452373568124842752,3453292832862501120,3501254041165084544,3501922067493606272,3501987282277848832,3503786766199868416,3521882906526249088,3523287498271504128,3523596186160340992,3526928874624208128,3528847870307147136,3529016744125888896,3549053213238728704,3552223689378645888,3552304366042386304,3552845261339955200,3556190937783724928,3559194558736936064,3559496413333223936,3562398363821309440,3564743244166615552,3565057738851555840,3565562247185973248,3566532561902107648,3566532561902107904,3578787203109091968,3593022683233287424,3598349645270793984,3600899515814587904,3601509985286515072,3601778888894719616,3602129564384942848,3615854321357005824,3615906062827657344,3616128542133976064,3637209066255998592,3638148805100101888,3638333660492606976,3638457669084084224,3645620879964502016,3646374187163652608,3647396385084826112,3648528607183584640,3650426742210702976,3650552739370519680,3650615686411758080,3650730589671206528,3651028381228835456,3651305423799211008,3651429943491857152,3651747667992161920,3652390092020688128,3652865390281317120,3653507298913645056,3653854572788715520,3653865018149220608,3653928313083020416,3654216312114820352,3654573962630823040,3654870693331906304,3656469211440196736,3656970966699043328,3657684515386065152,3658513620167922816,3659871753251882496,3659914737283635840,3660356573454829824,3660622479174640640,3661843349398271232,3661954022115425536,3662779171232754688,3663496155893287040,3663664003222454528,3663900436870097664,3663919296071216000,3666683987994884992,3667514634669861120,3667855139676395520,3667872216467245312,3669065354086975872,3669328034286134016,3669573053580577536,3669910500571362688,3669936102871625600,3671751946324947584,3672615917651101952,3672656947473710592,3673479966286848640,3674290886176428160,3674426504064071936,3674476639217656576,3674885584528395520,3676222491884053376,3682412703692940032,3682469122383597056,3682835848166848896,3683074094297008128,3683519503881169920,3683627840135415808,3684329706511747840,3684741301817503872,3684909561456453248,3685203985758805120,3686053976966539136,3686476533029004928,3687414210289577344,3688402091422432128,3688697924474802176,3690395231125833600,3690709554012428416,3691095100341397632,3691685882071367936,3693488019896536704,3693624423762089216,3694399755554510720,3694595296825212288,3695495693769418112,3695839596094423296,3696633688303809664,3696778892558087552,3697125646741428224,3698040337339931776,3698872156539379968,3698912834175717120,3699324841795942272,3701701509194344576,3704392873141718912,3704492619460777472,3704568344029167744,3704624487842182656,3705070756419217408,3705109338109763712,3706548701910085760,3707668520142907392,3707975009009227648,3708253980021111168,3708366954839619328,3708578473389974528,3708799608370516736,3708970651148026112,3709565658737443200,3709743195505951744,3710931217819026560,3711214067185666560,3711954863144745344,3712709304215673088,3712812452150011648,3714266139665215488,3714914271705535360,3715484707786679296,3715726703424296064,3717651845204820864,3718579386342034816,3719493011785709056,3721002778689505024,3721801956139414400,3722192007889372800,3722405308849506944,3722502856146928256,3723164143671997952,3724064162658265088,3724079384022987136,3724485756648081920,3724610104541796096,3725166869742133248,3725279157367006464,3725570772761744384,3726954405066019840,3727155340815968256,3728063945441922176,3728074738695246336,3728216571400055040,3729759568466399488,3731380729641520128,3731530843043705216,3731848189586855040,3731853549706103680,3732225807407269888,3732829885967805056,3733305179932909312,3733613970901618816,3733693754214402816,3733908713033609600,3734940188083120896,3734983412634263936,3736695803210593664,3737732878898142208,3738868468250890880,3739031165907785216,3739031268987206784,3740213514569182848,3741324506644465280,3741829590503668480,3742597358857731712,3743646365965636352,3743786072660949120,3744078023062858624,3744991824600484864,3745162141527947008,3746422113133965696,3778305545157841152,3781616827503753088,3781742343628799232,3785295862130129792,3785855994584716288,3790040460962601344,3790040465258127616,3790932787663209344,3790933504922574080,3791246075462610304,3791660630001113344,3791951961927535616,3793038176336413568,3793132257595347200,3793871056394569600,3794415245931016320,3795052348495488896,3796601418644353536,3799009353404271488,3800902265750001664,3802622073735216128,3802627670077268864,3804239558419007744,3804627716087570560,3805542578481614464,3806277464565712256,3806529291383459456,3808536101967194368,3808776242178788736,3809230860172408320,3809758930696054784,3810099989754827136,3810124316449529984,3810397128476695424,3810416820902223616,3810933247769901696,3812046945673961856,3813093066563138304,3814445324131846272,3815105615223622144,3815200997858084480,3815759279181364864,3816821334399520128,3817262208497857024,3828635144458312960,3829366074878263168,3830623164560911872,3830680854561582464,3830990156631488128,3831369659941271424,3831461159924430080,3832119767389994880,3832329434808415744,3832474810861495424,3833982271367482624,3834061470568945280,3834247326684614656,3834721387994870784,3834915417437497088,3835861439819152128,3835866563715176192,3835912017354065920,3838571117511435392,3840846114438361984,3840946062621774592,3842126835031738368,3843483151343659776,3843957354387777152,3844051225193053440,3844209481853063296,3844228070471625216,3844512328587119616,3845116029189633280,3846462656056088448,3846557660732848384,3847084567320260864,3847322577227869952,3848159236857539584,3848338216734554112,3848782522511323264,3850056169293341824,3850381865253059328,3851443134492197888,3851943584081217408,3852194891207625344,3853660510143027200,3854183431001210112,3854286918237953536,3855631797052747136,3855792630692650112,3855932749706134272,3856005626711298560,3856950175919062144,3857118439852847744,3858586012997916160,3859871342090899328,3860382404543853952,3860751256335185408,3861407596057614080,3862099876068198656,3862830334039617408,3862858165427681536,3863022297602869632,3863287692225892736,3863559855709416192,3863974783909961600,3864754303293985792,3865951435233552896,3866845986727270400,3870346934829606528,3870354528331257984,3870767467961813248,3870931050381491072,3871736820606012928,3874412413432643328,3874412413432647680,3875197155497209472,3875651975353757440,3875652014008894720,3875789001991057536,3876580134966772224,3876618892751168000,3876834259591671040,3877153599000231552,3878807333207905536,3879148010013223680,3880219037418284160,3881473820703514752,3882547566823086848,3882611201058534400,3883429134630276736,3883495444630204672,3883746133280920448,3884462057084493824,3884610594233452544,3884662713661807616,3884899559633556224,3885315003230198016,3886816622581111552,3887433414244718464,3888723386196630784,3888929097950924416,3889004478921783296,3889099689757032448,3890141958060313984,3890322415406225408,3891115064506627840,3892281367169398016,3892524535332945280,3893651289938122496,3893925965982400384,3894294435521417344,3894780007343533184,3894868518029804032,3894911089745430656,3895853993981558400,3896442675083660288,3897015868534544000,3898536321316235136,3899444135668572288,3899809757644549632,3899975238440888064,3900013579612540672,3901657108978035328,3902183809407583872,3902583722402774784,3902697968532578048,3902698243410506112,3903151246497510784,3904294940452344576,3904628406009296896,3905035495893288448,3905186270720273152,3905501688823640832,3905533368502667520,3905534918987016576,3905833810054930816,3905948159264050432,3907438925232641024,3907860278704120320,3908299636678454784,3909918156449969408,3910338303034941824,3910941183300119808,3911460633824772224,3912412050684471680,3912968163050466560,3913666009336762624,3914160793864337280,3914169864834617984,3914686699729551744,3915026861134449664,3915111042492943360,3917662149987657472,3917712246486088064,3917811443050820736,3918024679587352960,3918457681010093824,3919950955240152832,3920078086272085888,3921044866230104576,3921657294207109504,3921843871882038016,3922740248735399168,3924625975601987968,3926968661219149184,3927653893186082176,3928060510626899200,3928077827935031168,3928101738016062336,3928119295842399360,3929166408868873344,3929872707650824320,3930500666228897536,3930720057454344960,3930720263611949056,3930922058356114560,3931506719368273792,3932607193069290752,3933386712453327360,3934289587593617664,3934531063539402752,3935942939548822272,3936036638555796224,3936094878312557184,3936265787946147712,3936440262402010368,3936999437080327040,3937174942327932544,3937174946624964224,3937959615673187712,3938160280840704000,3940853397134158848,3940955205038857728,3941103359935056640,3941482519648231296,3941619958601686016,3942390956770954240,3942685179209931008,3943650619138622848,3944082692848953088,3944400490365194368,3944621251683633024,3944830674288852352,3945304834382891008,3947104533054775168,3947604806549970560,3948718856642651904,3949023868040468224,3949197105545275264,3949551801124462976,3951154923437922176,3951290541324811520,3951770649948609920,3952184341199008384,3952478391839564928,3953539321776133632,3954167379139173632,3954947379559880064,3956630388264212992,3956905747207784576,3957067310993650048,3958861915832531328,3958889751516035200,3959991973267755904,3960460743178289152,3960660407621272576,3961323069532233984,3961591831405665280,3962123479637203200,3962125442437204096,3962296416495937152,3964744479135122560,3967110520783248640,3967492742808436864,3968318334306923264,3969544976966357504,3971863297233655040,3972360517007764096,3972493871447452032,3973944741464447616,3974116235214146432,3974607884414675584,3974868644764927744,3976193895578807296,3976920329166692736,3977279976843578752,3978862277154958592,3978988652273088128,3979070527235441792,3979704120809463808,3979751266665795456,3980784841955866624,3980865789203927680,3981718048154691712,3982007636324256000,3982760934933535616,3983606596814071680,3984076534955978880,3984813379545559424,3987528623509801984,3988212592756945152,3989101543613421184,3989883777417007232,3990494251184010496,3991040498009369856,3992108467396622464,3993006940195159552,3995295401850242688,3995663944403608448,3995728953028752768,3996172640334271872,3998086241534550144,4001074538044978816,4001128414114602880,4001466277717538432,4001862578644425344,4002617427736191872,4008511467191955840,4010090670831602432,4010119017615780608,4010592357371458944,4011194202932877696,4011709259321463808,4012048699177439744,4012449333727266688,4012697887778691328,4013798537573517568,4013877015215909632,4014667186119082624,4015547856277853824,4015679213557387264,4015951102167164032,4016142868162408576,4016204440813547520,4016670251491467520,4017476365313408512,4019392852735219840,4019458647338779648,4019734594693512704,4020794214664565120,4021565827014024576,4022081429247669632,4024105492715825152,4026018956480400256,4027510375284724608,4028120776036373760,4030292070983358208,4031339076635734528,4031828157446980608,4032338708799508096,4032410520652747648,4034132866962718848,4034516424721922560,4035180770266708992,4095056359580563328,4140966708116861440,4141621639101946880,4190734280885088128,4195196996718089472,4195385769128825472,4208475450751790848,4211784160426521344,4226568193761131520,4226663404595171968,4227320672031546240,4227915744043939584,4228210894197669760,4228397712388844800,4228576550540445184,4228576554828200320,4229400153462745472,4229720527957915904,4230332092645516672,4230380819051252736,4232108426695238272,4233202195952355712,4235280071072332672,4236208432541189376,4237044301894347392,4237555506083389568,4237555510381371136,4239587888920128000,4240366136980012416,4246381595156273792,4250205872691106176,4250205872691106560,4250461749665556224,4276845523320382976,4277352024517980672,4299191688396959488,4299397713684601088,4321498378443922816,4321547066187820800,4328933138625682432,4329047998934483968,4331694970097765120,4334670557801027840,4337281421174823168,4337798810117194112,4338593516506343424,4339432164702764800,4340322876499078400,4341270792960082304,4351263356495106176,4352559818142869760,4352651700380171392,4354353293406211200,4357309158618673024,4358066107952246784,4358132250448454272,4358305011212780032,4365752926885263872,4366155146278913408,4366231386241199360,4368168107254401408,4368448074699014528,4371734308793396992,4372278631477380224,4372558083524803072,4372725209292759936,4374362824485469440,4376670218359261056,4376670222653381376,4377071681835414528,4379812558164849408,4380950067364935424,4382453172774669952,4384015024746850432,4384056565671592576,4385909109622569088,4385911549163997184,4386278128914548352,4387094619380065280,4387094623673782656,4388808448767668864,4388927887517644544,4394367510839007872,4402794756366307328,4403059253334693248,4403335028894782464,4403541148665907456,4403768373911022080,4403922924014755456,4404459928069184896,4405080812841630592,4405513672526728576,4406242000198708736,4406401906122017536,4406631841491803520,4406683522836435968,4406733207018066944,4407614499946419200,4408409820512876928,4409340076070042496,4409986554547744640,4411572123331402880,4412554051637282560,4413133013228067456,4414890277623163264,4414967307861766016,4415021737982254080,4415701304886958848,4415725940820061824,4415799779897925760,4416547310366442752,4416948833973175168,4417226117061248256,4417527967363383680,4420803996617732352,4421694158655153664,4422618092019322880,4422929253811220224,4423768662219609088,4423790579436883072,4423803223819720704,4424031479858305408,4424957298712211840,4425202936481819520,4425632987265111680,4425884324454250752,4426297740826647680,4426414946193200128,4426435871273922560,4427021425634792832,4427716454423219968,4429106481934491136,4429655653631543552,4430790968108076544,4431665212995124608,4434273946068357248,4436905352274528896,4437515851806126208,4437884055061048064,4437908587915262848,4438653541403499776,4438770158349772672,4438822655736145408,4439766762564880512,4440057823911572992,4443586358581794176,4444590625015876864,4446506356529203968,4446863869601988096,4446890665906313088,4447039585308297088,4447152865064276736,4448378171991893888,4448453209363580160,4449057326579614592,4449495546386664832,4449576983265175424,4449634256649531776,4449818459207085696,4450425359563998720,4450425359563998848,4451097882722882432,4451552182885460480,4451874958268328960,4452997701376633728,4453746571874680064,4453933690711829504,4454017257893306496,4454109681291473536,4454328930782860800,4454334020319210624,4454676827432347904,4455205894384768256,4455760769799431040,4456018914514270592,4458207634145130368,4458251064854338176,4458886273338645888,4459666716138776576,4459687645513849984,4460086458999242368,4460327252045435648,4461228611063376128,4461401745486070144,4461433528244756480,4461739604794187520,4461972907417918336,4462612140287443840,4462895505049732608,4463261431966304768,4463664815294536192,4463681655863346816,4464010508621809664,4464073180784561408,4464512882357505280,4464512886654172160,4465105351622152704,4465939601772584448,4466388790929771904,4466984859374027904,4467229882964819456,4467742388527832192,4469700519955285760,4470233817461336704,4472775712243958272,4473632403597115776,4473925114207017728,4486694709112801536,4487104135457345920,4487371007546472832,4488537864259886336,4490300553197280256,4490567017265616128,4490607183799536896,4491465107810964096,4491748511228743808,4494877446445481472,4497001870409825280,4517521407404432512,4518509486084557312,4518917168694695168,4520333442759039360,4525569007873380736,4530172280800983808,4540774733984289792,4541415680545226624,4544277782323169152,4544925596536285056,4549622027311531904,4556553928430812672,4557059845518172288,4558191071185210368,4558448219470271616,4559106586411037440,4559800992425812096,4560373769265200640,4561417003938888704,4561641991506408448,4562566234107439616,4564029851585380608,4565048312887877888,4565151422166297088,4565386996828484224,4566554472019360128,4566955999923881344,4567158653660872064,4568037330951578368,4568179481487112832,4568641929204497664,4569667429956444032,4569753531162638464,4570199658005551232,4570546317703725312,4570748524763671168,4571723134445705856,4572405140891735296,4572927688089973632,4573071139998034048,4573650303451549952,4574032765995786496,4574579974891723136,4574753392788056960,4576413655356340352,4577666681988285696,4577915996250307968,4578278903808759040,4578464515115212672,4581312593826445184,4581383928942599296,4582366200845256960,4582511267660632832,4595011409196652544,4598266758185956864,4598568230530318592,4598775557191664384,4598830670211908864,4598830738931385984,4598853961817753472,4598979924619296384,4599036137150118656,4599998179762051200,4600107233275449088,4601940394036184192,4602478059518965632,4610819951159775616,4610983121260366336,4611204329256349312,4611543459874840832,5112787862966795136,5113119911182752768,5131505910262456448,5131607855606329344,5131731035268115584,5131731039563630720,5138313850039513472,5141213292265583616,5142004322162569984,5142174201004781440,5142197118950177280,5142856894646708864,5143835352622910720,5145458369221838720,5147930591051748480,5147997450808347648,5149431866805198848,5149797798018991232,5149803639174596224,5149828343826748800,5151008051083531392,5167510797199003904,5167525434447523712,5168944835239726080,5168947721457740288,5169733318219971968,5174110233491949312,5178957121265572608,5179920984941752448,5180178266367085312,5180494685197598464,5188044687948351872,5763341263596849920,5763373596110853248,5764001760847057280,5764027015254831488,5764107760640582656,5764485618978306176,5764602334714811904,6244198414713179392,6248279939316190464,6249697450322286720,6250213984568447872,6250372378664759808,6261001052630781312,6262044283006050432,6265082680310200448,6265877455415860224,6276520070839642496,6279455526367838592,6279493528239353728,6284524274971664000,6314498920450406016,6318882711964895872,6319113776910198656,6319467136754514176,6319765821665661440,6319913126159509888,6331610452409422080,6333328778630560896,6338811188419479040,6340103831841550080,6860084802130167936,6861525956933302912,6861930164895495424,6875432476920696448,6875432476922523520,6875737247801369344,6879524790480960896,6879638761736608768,6879695313566067712,6879784584465884544,6880450613630457472,6880851931079280512,6886074198993703168,6886271973655421824,6886287332455279104,6887386745296346624,6889315284394862336,6889320030334437376,6891429026779675520,6892132782942459136,6895690424612412416,6897289256251743744,6897451292484010624,6898453913943607040,6898455661995166336,6898489884295407488,6898489884295412352,6898521877506794880,6907031749613795968,6907079269131974912,6907108371830928768,6909994246262080000,6913080751488667520,6913362119090316416,6913719808262954880,6913880096442179200,6914243308941804288,6915782870033442688,6915911134936982784,6916168897398516096,6916942197670351872)
    """

    # Perform TAP query
    job = Gaia.launch_job_async(query)  # This runs for <30min
    results = job.get_results()

    # Save to file
    results.write(filename, format="votable", overwrite=True)
    print("Data download complete.")
else:
    print("Data file found. Skipping Gaia query.")

INFO: Query finished. [astroquery.utils.tap.core]
Data download complete.
